# AE Reconstruction Comparison

Notebook wrapper for the MLP vs atlas-free CNN autoencoder reconstruction comparison. The evaluation code lives in `atlas_free_cnn.evaluation.compare_ae_reconstruction`; this notebook only configures and calls it.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "neurovlm").exists() and (candidate / "experiments" / "3dcnn" / "atlas_free_cnn").exists():
            return candidate
    raise RuntimeError("Could not find repo root. Start Jupyter from the neurovlm repo or update this cell.")

REPO_ROOT = find_repo_root()
THREEDCNN = REPO_ROOT / "experiments" / "3dcnn"
MODEL_COMPARISON_DIR = THREEDCNN / "model_comparison"
for path in [REPO_ROOT / "src", THREEDCNN, MODEL_COMPARISON_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

REPO_ROOT

In [ ]:
import pandas as pd

from atlas_free_cnn.evaluation.compare_ae_reconstruction import (
    AE_MODEL_IDS,
    DATASETS,
    DEFAULT_OUTPUT_DIR,
    run_comparison,
)

list(DATASETS), list(AE_MODEL_IDS)

## Configure

Set `RUN_MODE` below: `"quick"` is a fast sanity check (PubMed only, MLP plus
one CNN model, two samples); `"full"` is the actual comparison across all
datasets/models (downloads CNN + MLP checkpoints and the unified test split
on first run, then reads from local HF cache). Everything below -- Run
Comparison, Inspect Outputs, Visualize Results -- uses whichever mode you
pick here.

In [ ]:
# RUN_MODE:
#   "quick" - fast sanity check: PubMed only, MLP + one CNN model, two samples.
#   "full"  - the actual comparison across all datasets/models.
RUN_MODE = "full"

if RUN_MODE == "quick":
    DATASETS_TO_RUN = ["pubmed"]
    MODELS = ["mlp_neurovlm", "cnn_ae_mixed"]
    LIMIT = 2
elif RUN_MODE == "full":
    DATASETS_TO_RUN = ["pubmed", "neurovault", "nilearn"]
    MODELS = list(AE_MODEL_IDS)
    LIMIT = 16
else:
    raise ValueError(f"Unknown RUN_MODE {RUN_MODE!r}; expected 'quick' or 'full'")

DEVICE = "cpu"
BATCH_SIZE = 8
SKIP_VOXEL_AUROC = False
SKIP_MLP_FLAT = False

preferred_test_jsonl = REPO_ROOT / "experiments" / "3dcnn" / "atlas_free_cnn" / "cache" / "unified_jsonl" / "splits" / "test.jsonl"
TEST_JSONL = preferred_test_jsonl if preferred_test_jsonl.exists() else None

OUTPUT_DIR = REPO_ROOT / DEFAULT_OUTPUT_DIR

{
    "run_mode": RUN_MODE,
    "datasets": DATASETS_TO_RUN,
    "models": MODELS,
    "limit": LIMIT,
    "device": DEVICE,
    "test_jsonl": str(TEST_JSONL) if TEST_JSONL else None,
    "output_dir": str(OUTPUT_DIR),
}

## Run Comparison

In [ ]:
result = run_comparison(
    datasets=DATASETS_TO_RUN,
    models=MODELS,
    limit=LIMIT,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
    test_jsonl=TEST_JSONL,
    batch_size=BATCH_SIZE,
    include_voxel_auroc=not SKIP_VOXEL_AUROC,
    include_mlp_flat=not SKIP_MLP_FLAT,
)

result

## Inspect Outputs

In [ ]:
summary = pd.read_csv(result["summary_csv_path"])
summary

In [ ]:
by_sample = pd.read_csv(result["by_sample_path"])
by_sample.head(20)

## Visualize Results

Grouped bars compare mean metrics across models within each dataset. Color is
fixed per **model family** (MLP, CNN mixed baseline, CNN domain-specialized)
across every chart in this notebook and in the other two comparison
notebooks -- the same model always reads as the same color. Bars are
annotated with their value; a missing bar means that model/dataset
combination produced no supported rows (see the coverage table below for
why, e.g. `missing_checkpoint`).

The MLP autoencoder was trained on **binary** PubMed activation masks, so on
**PubMed** it is evaluated with the main package's dedicated flat resource
(`mlp_masker_flatmap`). NeuroVault and Nilearn have no such resource, but the
atlas-free CNN's packed `(36, 45, 38)` volumes turn out to be crops of the
*exact same MNI152 4mm grid* the MLP masker uses (see
`atlas_free_cnn.evaluation.mlp_masker_bridge`) -- so those volumes are
converted into MLP masker-flat space with a boolean crop index (no
resampling), binarized to match the MLP's training distribution, and
evaluated the same way as PubMed. This shows up as a second comparison
space, `atlas_free_volume_via_mlp_masker_crop`, in the coverage table and
plots below.


In [ ]:
import matplotlib.pyplot as plt
import plotting_utils as pu

coverage_df = summary.copy()
if not coverage_df.empty:
    coverage_df["status"] = coverage_df["n_supported"].gt(0).map({True: "ok", False: "unsupported"})
pu.coverage_table(coverage_df)

In [ ]:
metrics_to_plot = [
    ("mse_mean", "Mean squared error", "MSE", "{:.3f}"),
    ("mae_mean", "Mean absolute error", "MAE", "{:.3f}"),
    ("spatial_corr_mean", "Spatial correlation", "Pearson r", "{:.2f}"),
    ("top5_dice_mean", "Top-5% Dice", "Dice", "{:.2f}"),
]
fig_metrics, axes = plt.subplots(1, len(metrics_to_plot), figsize=(21, 4.5))
for index, (column, title, ylabel, value_fmt) in enumerate(metrics_to_plot):
    pu.grouped_bar(
        summary, category_col="dataset", series_col="model_id", value_col=column,
        ax=axes[index], title=title, ylabel=ylabel, value_fmt=value_fmt,
        show_legend=index == len(metrics_to_plot) - 1,
    )
fig_metrics.suptitle("Autoencoder Reconstruction Quality by Dataset and Model", x=0.01, ha="left", fontsize=12)
fig_metrics.tight_layout(rect=(0, 0, 0.9, 0.95))

In [ ]:
REPORT_ASSETS_DIR = OUTPUT_DIR / "report_assets" / "ae_reconstruction"

report_manifest = pu.save_report_assets(
    REPORT_ASSETS_DIR,
    figures={"ae_reconstruction_metrics": fig_metrics},
    dataframes={"ae_reconstruction_summary": summary, "ae_reconstruction_by_sample": by_sample},
)
report_manifest